In [ ]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, Engine
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import HTML

load_dotenv("../.env")

# start psql engine to grab data from database
def get_engine() -> Engine:
    '''
    Depending on .env, it will load localhost or online DB
    priority is online DB
    '''
    DATABASE_URL = os.environ.get("DATABASE_URL")

    if not DATABASE_URL:
        DB_USER = os.environ.get("DB_USER")
        DB_PASSWORD = os.environ.get("DB_PASSWORD")
        DB_HOST = os.environ.get("DB_HOST", "localhost")
        DB_PORT = os.environ.get("DB_PORT", "5432")
        DB_NAME = os.environ.get("DB_NAME")

        if not all([DB_USER, DB_PASSWORD, DB_NAME]):
            raise RuntimeError(
                "Set either DATABASE_URL, or DB_USER/DB_PASSWORD/DB_NAME in .env"
            )

        DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

    else:
        DATABASE_URL = f"postgresql+psycopg2://{DATABASE_URL}"

    return create_engine(DATABASE_URL)

engine = get_engine()
print(engine.url)

In [ ]:
query = """
SELECT *
FROM coffee_production
"""

coffee_prod_df = pd.read_sql(query, engine)

df1 = coffee_prod_df

df1 = df1['1990'].apply(lambda x: x/1000000)
df1.head()

In [ ]:
# create new df with year and total production to match co2 table
year_cols = [col for col in coffee_prod_df.columns if str(col) in [str(y) for y in range(1990, 2020)]]

# 2. Sum down the rows (axis=0) and convert to a new DataFrame
coffee_total_by_year_df = coffee_prod_df[year_cols].sum().reset_index()

# 3. Rename columns, year is str at this moment
coffee_total_by_year_df.columns = ['year', 'total_production']

coffee_total_by_year_df['year'] = coffee_total_by_year_df['year'].astype(int)
coffee_total_by_year_df.info()


In [ ]:
query = """
SELECT *
FROM coffee_export
"""
export_df = pd.read_sql(query, engine)

query = """
SELECT *
FROM coffee_import
"""
import_df = pd.read_sql(query, engine)

query = """
SELECT *
FROM coffee_domestic_consumption
"""
dom_consumption_df = pd.read_sql(query, engine)

query = """
SELECT *
FROM coffee_importers_consumption
"""
impt_consumption_df = pd.read_sql(query, engine)

export_df.shape
#import_df.shape
#dom_consumption_df.shape
#impt_consumption_df.shape


In [ ]:
query = """
SELECT *
FROM global_co2
WHERE year BETWEEN 1990 and 2019
"""

co2_df = pd.read_sql(query, engine)

# cast year as int so that it works during merge with coffee_production
co2_df['year'] = co2_df['year'].astype(int)
co2_df.info()

In [ ]:
coffee_co2_df = coffee_total_by_year_df.merge(co2_df, on='year', how='left')
coffee_co2_df = coffee_co2_df[['year','total_production','co2_concentration']]
coffee_co2_df.head()

sns.scatterplot(data=coffee_co2_df,x='year',y='total_production',hue='co2_concentration')
plt.show()


In [ ]:
coffee_robusta = coffee_prod_df[coffee_prod_df['coffee_type'].isin(['Robusta'])]
coffee_arabica = coffee_prod_df[coffee_prod_df['coffee_type'].isin(['Arabica'])]
coffee_mix = coffee_prod_df[(~coffee_prod_df['coffee_type'].isin(['Robusta'])) &
                            (~coffee_prod_df['coffee_type'].isin(['Arabica']))]
coffee_mix['coffee_type'].unique()

In [ ]:


# Your DataFrame 'df' with columns: 'Country' and 'Total_Production'
fig = px.choropleth(
    coffee_mix,
    locations="country_code",      # ISO 3-letter codes (e.g., 'USA', 'DEU', 'BRA')
    color="total_production",      # Column controlling color intensity
    color_continuous_scale="Viridis",
    hover_name="country",          # Shows full country name in bold hover title
    hover_data={
        "country_code": False,      # Hides the code line from hover body
        "total_production": ":,f"   # Keeps production number (formatted with commas)
    },# Info shown on hover
    title="Total Production Volume by Country"
)

fig.show()

In [ ]:
year_cols = [str(y) for y in range(1990, 2020)]  # Or [1990, 1991, ...] if integers

# Reshape from wide to long
long_df = coffee_prod_df.melt(
    id_vars=['country'],           # The column to keep fixed (change if named differently)
    value_vars=year_cols,          # The year columns to unpivot
    var_name='year',               # New column name for the years
    value_name='production'        # New column name for the production numbers
)

# 1. Clean types and rank production per year
long_df['production'] = pd.to_numeric(long_df['production'], errors='coerce').fillna(0)
long_df['rank'] = long_df.groupby('year')['production'].rank(method='first', ascending=False)

# 2. Keep only the top 10 for each year
top10_df = long_df[long_df['rank'] <= 10].copy()

# 3. Create the animated bar chart
fig = px.bar(
    top10_df,
    x="production",
    y="rank",
    color="country",
    text="country",
    animation_frame="year",
    animation_group="country",
    orientation="h",
    title="Top 10 Coffee Producers Over Time",
    range_x=[0, top10_df['production'].max() * 1.1],
    height=800,   # Height in pixels (default is ~450)
    width=1200    # Width in pixels (default fits container width)
)

# 4. Format layout (Rank 1 at the top)
fig.update_yaxes(autorange="reversed", title="Rank", dtick=1)
fig.update_traces(textposition="inside", texttemplate="%{text}")
fig.update_layout(showlegend=False, height=500)

fig.show()


In [ ]:
year_cols = [str(y) for y in range(1990, 2020)]
export_df = coffee_prod_df[["country"] + year_cols]

long_df = export_df.melt(id_vars="country", var_name="year", value_name="volume")
long_df["volume"] = long_df["volume"] / 1_000_000
long_df.to_csv("coffee_production_long.csv", index=False)

In [ ]:
import numpy as np
from itertools import cycle
from matplotlib import animation, ticker

plt.rcParams["animation.embed_limit"] = 50  # MB, raise if you get a size error

# %% Cell 2 - prep data
YEAR_START, YEAR_END = 1990, 2019
year_cols = [str(y) for y in range(YEAR_START, YEAR_END + 1)]

df = (
    coffee_prod_df.set_index("country")[year_cols]
    .rename(columns=int)
    .sort_index(axis=1)
    .div(1_000_000)
)

# %% Cell 3 - reshape + smooth
df_t = df.T
df_t.index.name = "year"
df_t.index = df_t.index.astype(float)

STEPS_PER_YEAR = 20  # increase for smoother/slower animation

years = df_t.index.to_numpy()
new_index = np.concatenate(
    [np.linspace(y0, y1, STEPS_PER_YEAR, endpoint=False) for y0, y1 in zip(years, years[1:])]
    + [years[-1:]]
)

df_smooth = df_t.reindex(df_t.index.union(new_index)).interpolate(method="index").loc[new_index]

# %% Cell 4 - animation setup
N_BARS = 10
country_colors = dict(zip(df.index, cycle(plt.cm.tab20.colors)))

fig, ax = plt.subplots(figsize=(12, 7))
fig.subplots_adjust(left=0.22, right=0.95, top=0.9, bottom=0.08)

def draw_frame(i):
    ax.clear()
    row = df_smooth.iloc[i]
    top10 = row.nlargest(N_BARS).sort_values()

    bars = ax.barh(top10.index, top10.values, color=[country_colors[c] for c in top10.index])

    for bar, val in zip(bars, top10.values):
        ax.text(
            val + row.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f}", va="center", ha="left", fontsize=10,
        )

    ax.text(
        0.95, 0.05, str(round(df_smooth.index[i])),
        transform=ax.transAxes, fontsize=40, fontweight="bold",
        ha="right", va="bottom", color="grey", alpha=0.4,
    )

    ax.set_xlim(0, df_smooth.values.max() * 1.15)
    ax.set_title("Top 10 Coffee Producing Countries", fontsize=16, fontweight="bold", loc="left")
    ax.set_xlabel("Production volume")
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    ax.set_ylabel("")

anim = animation.FuncAnimation(fig, draw_frame, frames=len(df_smooth), interval=100, repeat=False)
plt.close(fig)

# %% Cell 5
HTML(anim.to_jshtml())

# %% Cell 6 - optional export
# anim.save("coffee_bar_chart_race.gif", writer="pillow", fps=15)
# anim.save("coffee_bar_chart_race.mp4", writer="ffmpeg", fps=15)